# Task 2: Quantitative Analysis Using PyNance and TA-Lib

## Objective
Load historical stock price data, compute financial technical indicators, and visualize the results to understand market behavior.

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import talib
import pynance as pn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## Load and Inspect Stock Price Data

In [ ]:
# Load the AAPL stock data
df = pd.read_csv('../newsData/AAPL.csv', parse_dates=['Date'], index_col='Date')

# Display the first few rows
print("Data shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

## Clean and Prepare Data

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Ensure columns are numeric
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Handle any remaining missing values (if any) by forward fill
df = df.fillna(method='ffill')

# Sort by date
df = df.sort_index()

print("\nData after cleaning:")
print("Shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())
print("Missing values after cleaning:", df.isnull().sum().sum())

## Compute Moving Averages (SMA, EMA) with TA-Lib

In [ ]:
# Calculate Simple Moving Averages
df['SMA_20'] = talib.SMA(df['Close'], timeperiod=20)
df['SMA_50'] = talib.SMA(df['Close'], timeperiod=50)

# Calculate Exponential Moving Averages
df['EMA_20'] = talib.EMA(df['Close'], timeperiod=20)
df['EMA_50'] = talib.EMA(df['Close'], timeperiod=50)

print("Moving Averages computed:")
print(df[['Close', 'SMA_20', 'SMA_50', 'EMA_20', 'EMA_50']].tail())

## Compute RSI with TA-Lib

In [ ]:
# Calculate Relative Strength Index
df['RSI'] = talib.RSI(df['Close'], timeperiod=14)

print("RSI computed:")
print(df[['Close', 'RSI']].tail())

## Compute MACD with TA-Lib

In [ ]:
# Calculate MACD
df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(df['Close'], fastperiod=12, slowperiod=26, signalperiod=9)

print("MACD computed:")
print(df[['Close', 'MACD', 'MACD_signal', 'MACD_hist']].tail())

## Apply PyNance for Additional Financial Metrics

In [ ]:
# Calculate daily returns
df['Daily_Return'] = df['Close'].pct_change()

# Calculate rolling volatility (30-day)
df['Volatility_30'] = df['Daily_Return'].rolling(window=30).std() * np.sqrt(252)  # Annualized

# Calculate Sharpe ratio (assuming risk-free rate of 0.02)
risk_free_rate = 0.02
df['Sharpe_Ratio'] = (df['Daily_Return'].rolling(window=30).mean() - risk_free_rate/252) / df['Volatility_30']

print("Additional metrics computed:")
print(df[['Close', 'Daily_Return', 'Volatility_30', 'Sharpe_Ratio']].tail())

## Visualize Price with Moving Averages

In [ ]:
# Plot closing prices with moving averages
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='Close Price', alpha=0.7)
plt.plot(df.index, df['SMA_20'], label='SMA 20', linewidth=2)
plt.plot(df.index, df['SMA_50'], label='SMA 50', linewidth=2)
plt.plot(df.index, df['EMA_20'], label='EMA 20', linewidth=2, linestyle='--')
plt.plot(df.index, df['EMA_50'], label='EMA 50', linewidth=2, linestyle='--')
plt.title('AAPL Stock Price with Moving Averages')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Visualize RSI and MACD